# Manila Barangay Flood Risk Complete Pipeline

## 1. Project overview and methodology summary

**Thesis title focus:** Barangay-level flood risk priority classification for the City of Manila using a multi-criteria Disaster Prioritization Index (DPI) and supervised machine learning.

## Study unit and coverage
- **Unit of analysis:** barangay
- **Coverage:** all **897** barangays of the City of Manila (total enumeration, not sampling)
- If the final integrated dataset does not contain exactly 897 barangays, the notebook raises an error

## Pipeline stages (merged from two workflows)
1. **Merged dataset** — spatial flood CSV + PSA 2020 + PSA 2024 population + administrative attributes + elevation + engineered features
2. **Thesis ML** — CSI, Exposure, Vulnerability, entropy-weighted DPI, DPI risk classes, and RF / GB / MLP classification

## Updated methodology highlights
- Use **only** validated LiPAD **5-year** and **25-year** flood layers for DPI and ML (100-year excluded)
- Integrate **both** PSA 2020 and PSA 2024 populations
- Use **Population Change 2020–2024 (%)** (not annualized growth)
- Compute density with **PSA 2024** population
- Vulnerability uses density, population change, and reverse-normalized elevation (**Relative Population Density removed**)
- ML target is **DPI_Risk_Class** (index-derived priority classes, not historical flood outcomes)
- Preprocessing stays inside sklearn pipelines; model selection uses training CV only; held-out test is evaluated once

## Important interpretation note
Outputs are **planning-support** prototypes for barangay prioritization research. They are **not** official flood warnings, forecasts, or real-time inundation predictions.


## Files Needed to Run the Tabular Pipeline

Because the shape-to-CSV/geospatial processing was completed separately in QGIS, this notebook now requires only tabular inputs:

1. `MetroManila_Barangay_Flood_Area_Validated.csv` — QGIS-prepared flood area and flood percentage table.
2. `population-dataset.csv` — PSA 2024 barangay population estimates.
3. `NCR.xlsx` — PSA 2020 barangay population table.
4. `manila_final_base_dataset_NEW.csv` — administrative/base barangay metadata.
5. `manila_barangay_elevation_official.csv` — required QGIS Zonal Statistics output containing barangay-level DEM/DTM elevation statistics.

### Elevation Requirement

Elevation is treated as a vulnerability input. The elevation table must contain barangay-level mean elevation for all 897 Manila barangays. The expected QGIS fields are:

- `adm4_name`
- `Elev_count`
- `Elev_mean`
- `Elev_median`
- `Elev_min`
- `Elev_max`

These are standardized in the notebook as:

- `Barangay`
- `Elevation_Count`
- `Elevation_Mean`
- `Elevation_Median`
- `Elevation_Min`
- `Elevation_Max`

`Elevation_Mean` is used in the Vulnerability Score and is reverse-normalized because lower elevation corresponds to higher flood susceptibility.


## List of Output Produced by the Pipeline
manila_barangay_elevation.csv

manila_barangay_flood_spatial.csv

manila_barangay_flood_population_elevation_enriched.csv

manila_ml_ready_dataset.csv

manila_dpi_rankings.csv

manila_model_comparison.csv

manila_feature_importance.csv

best_model_pipeline.joblib


## 2. Library imports and configuration

**Purpose:** Import required libraries and set global analysis constants.

**Input:** None (environment packages).

**Output:** Configured Python session (display options, random state, expected barangay count, CRS, tolerances).

**Why needed:** Reproducible thesis runs require a fixed random state, consistent numeric handling, and explicit study parameters (897 barangays; EPSG:32651 for area).


In [1]:
from __future__ import annotations

import os
import re
import warnings
from pathlib import Path
from typing import Iterable, Optional

import joblib
import numpy as np
import pandas as pd
import geopandas as gpd

from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

EXPECTED_BARANGAY_COUNT = 897
AREA_EPSG = 32651  # UTM Zone 51N for Manila area calculations
CITY_FILTER = "CITY OF MANILA"
RANDOM_STATE = 42
EPS = 1e-6

EXPECTED_PSA_2020_TOTAL = 1_846_513
EXPECTED_PSA_2024_TOTAL = 1_902_590
EXPECTED_POP_CHANGE_ABS = 56_077
EXPECTED_POP_CHANGE_PCT = 3.04

KEEP_ONLY_NUMBERED_BARANGAYS = True
EXPECTED_MIN_BARANGAY_NO = 1
EXPECTED_MAX_BARANGAY_NO = 905
APPLY_HAZARD_ATTRIBUTE_FILTER = False
HAZARD_FILTER_RULES = {}

print("Libraries imported.")
print(f"Expected barangays: {EXPECTED_BARANGAY_COUNT}")
print(f"Area CRS: EPSG:{AREA_EPSG}")
print(f"Random state: {RANDOM_STATE}")


# Optional elevation extraction from NAMRIA DTM
try:
    import rasterio
    from rasterstats import zonal_stats
except ImportError:
    rasterio = None
    zonal_stats = None
    warnings.warn("Optional packages rasterio/rasterstats are unavailable. The notebook can still use a precomputed elevation CSV.")


Libraries imported.
Expected barangays: 897
Area CRS: EPSG:32651
Random state: 42


## 3. Input path configuration

**Purpose:** Define all required input and output paths in one place.

**Input:** User-editable paths to the QGIS-prepared flood table, population files, administrative base data, and required QGIS elevation table.

**Output:** Resolved `Path` objects used by every later stage.

**Why needed:** After paths are set correctly, the notebook should run top-to-bottom without further interaction.

This notebook assumes that spatial preprocessing was already completed in QGIS. Flood-area computation and elevation extraction are treated as upstream geospatial steps.

For area-based GIS calculations, EPSG:32651 was used where appropriate. The final CSV files are non-geometric tabular outputs.

In [2]:
# ---------------------------------------------------------------------------
# USER CONFIGURATION — TABULAR PIPELINE
# ---------------------------------------------------------------------------
# This notebook starts from CSV/XLSX tables exported from QGIS and other sources.
# It does not perform flood polygon intersections in this tabular workflow.

BASE_DIR = Path(".")  # set to your project folder if running outside the upload directory
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Required tabular inputs
FLOOD_CSV = BASE_DIR / "MetroManila_Barangay_Flood_Area_Validated.csv"
PSA_2024_CSV = BASE_DIR / "population-dataset-2024.csv"
PSA_2020_XLSX = BASE_DIR / "population-dataset-2020.xlsx"
BASE_ADMIN_CSV = BASE_DIR / "manila_final_base_dataset_NEW.csv"

# Required QGIS Zonal Statistics elevation output
# Expected source file from QGIS:
# adm4_name, Elev_count, Elev_mean, Elev_median, Elev_min, Elev_max
ELEVATION_CSV = BASE_DIR / "manila_barangay_elevation.csv"

# Elevation is required in the final thesis methodology
REQUIRE_ELEVATION = True

# Optional reference only; not required for this tabular run
DTM_PATH = BASE_DIR / "DTM_ManilaCity_UTM51_PRS92.tif"
BARANGAY_BOUNDARY_PATH = BASE_DIR / "manila_barangays.geojson"
COMPUTE_ELEVATION_FROM_DTM_IF_AVAILABLE = False

# Outputs
OUT_FLOOD_CSV = OUTPUT_DIR / "MetroManila_Barangay_Flood_Area_Validated.csv"
OUT_ENRICHED_CSV = OUTPUT_DIR / "manila-city-flood-population_enriched_NEW.csv"
OUT_ENRICHED_XLSX = OUTPUT_DIR / "manila-city-flood-population_enriched_NEW.xlsx"
OUT_ML_READY_CSV = OUTPUT_DIR / "manila_ml_ready_dataset.csv"
OUT_MODEL_COMPARISON_CSV = OUTPUT_DIR / "model_comparison_results.csv"
OUT_PREDICTIONS_CSV = OUTPUT_DIR / "barangay_flood_risk_predictions.csv"
OUT_BEST_MODEL_JOBLIB = OUTPUT_DIR / "best_flood_risk_model_pipeline.joblib"

print("BASE_DIR:", BASE_DIR.resolve())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())

for label, path in {
    "FLOOD_CSV": FLOOD_CSV,
    "PSA_2024_CSV": PSA_2024_CSV,
    "PSA_2020_XLSX": PSA_2020_XLSX,
    "BASE_ADMIN_CSV": BASE_ADMIN_CSV,
    "ELEVATION_CSV required": ELEVATION_CSV,
    "DTM_PATH reference only": DTM_PATH,
    "BARANGAY_BOUNDARY_PATH reference only": BARANGAY_BOUNDARY_PATH,
}.items():
    print(f"{label} exists:", path.exists(), "-", path)

if REQUIRE_ELEVATION and not ELEVATION_CSV.exists():
    raise FileNotFoundError(
        f"Required elevation file not found: {ELEVATION_CSV}. "
        "Use the QGIS Zonal Statistics output file or update ELEVATION_CSV."
    )

BASE_DIR: /content
OUTPUT_DIR: /content/outputs
FLOOD_CSV exists: True - MetroManila_Barangay_Flood_Area_Validated.csv
PSA_2024_CSV exists: True - population-dataset-2024.csv
PSA_2020_XLSX exists: True - population-dataset-2020.xlsx
BASE_ADMIN_CSV exists: True - manila_final_base_dataset_NEW.csv
ELEVATION_CSV required exists: True - manila_barangay_elevation.csv
DTM_PATH reference only exists: False - DTM_ManilaCity_UTM51_PRS92.tif
BARANGAY_BOUNDARY_PATH reference only exists: False - manila_barangays.geojson


## Shared helper functions

**Purpose:** Centralize reusable operations used across spatial, merge, index, and ML stages.

**Input / output:** Pure utility functions (name cleaning, normalization, entropy weights, validation summaries).

**Why needed:** Thesis reproducibility and readability; repeated formulas (min-max, reverse min-max, entropy) must be applied consistently.


In [3]:
def clean_barangay_name(x) -> str:
    """Standardize barangay names for merges (e.g., Brgy. 105 -> BARANGAY 105)."""
    x = str(x).upper().strip()
    x = re.sub(r"\s+", " ", x)
    x = x.replace("BRGY.", "BARANGAY").replace("BRGY", "BARANGAY")
    return x


def clean_barangay_name_strict(x) -> str:
    """Strict merge key: remove spaces and hyphens (BARANGAY 202-A -> BARANGAY202A)."""
    return clean_barangay_name(x).replace("-", "").replace(" ", "")


def extract_barangay_number(name) -> Optional[int]:
    if pd.isna(name):
        return None
    match = re.search(r"Barangay\s+(\d+)", str(name), flags=re.IGNORECASE)
    return int(match.group(1)) if match else None


def extract_number(text) -> int:
    match = re.search(r"(\d+)", str(text))
    return int(match.group()) if match else 0


def is_numbered_barangay(name) -> bool:
    if pd.isna(name):
        return False
    return bool(re.fullmatch(r"Barangay\s+\d+(-[A-Z])?", str(name).strip(), flags=re.IGNORECASE))


def find_first_existing_column(df: pd.DataFrame, candidates: list, label: str) -> str:
    col = next((c for c in candidates if c in df.columns), None)
    if not col:
        raise ValueError(f"Could not find {label} column among {list(df.columns)}")
    return col


def keep_existing(cols: Iterable[str], df: pd.DataFrame) -> list:
    return [c for c in cols if c in df.columns]


def safe_numeric(df: pd.DataFrame, cols: Iterable[str]) -> pd.DataFrame:
    for col in cols:
        if col not in df.columns:
            continue
        if df[col].dtype == object:
            df[col] = (
                df[col].astype(str)
                .str.replace(",", "", regex=False)
                .str.replace("%", "", regex=False)
            )
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def minmax_normalize(series: pd.Series) -> pd.Series:
    """Min-max normalization: higher raw value -> higher normalized risk contribution."""
    series = pd.to_numeric(series, errors="coerce")
    min_val, max_val = series.min(), series.max()
    if pd.isna(min_val) or pd.isna(max_val) or max_val == min_val:
        return pd.Series(0.0, index=series.index)
    return (series - min_val) / (max_val - min_val)


def minmax_reverse(series: pd.Series) -> pd.Series:
    """Reverse min-max: lower raw value -> higher risk (used for elevation)."""
    series = pd.to_numeric(series, errors="coerce")
    min_val, max_val = series.min(), series.max()
    if pd.isna(min_val) or pd.isna(max_val) or max_val == min_val:
        return pd.Series(0.0, index=series.index)
    return (max_val - series) / (max_val - min_val)

def standardize_elevation_columns(elevation_df: pd.DataFrame) -> pd.DataFrame:
    """
    Standardize QGIS Zonal Statistics elevation column names.

    Accepts common QGIS/export variants and converts them to:
    Barangay, Elevation_Count, Elevation_Mean, Elevation_Median,
    Elevation_Min, Elevation_Max.
    """
    elevation_df = elevation_df.copy()
    elevation_df.columns = [str(c).strip() for c in elevation_df.columns]

    rename_map = {
        "adm4_name": "Barangay",
        "ADM4_NAME": "Barangay",
        "Barangay Name": "Barangay",
        "barangay": "Barangay",
        "Barangay": "Barangay",

        "Elev_count": "Elevation_Count",
        "ELEV_COUNT": "Elevation_Count",
        "Count": "Elevation_Count",
        "count": "Elevation_Count",

        "Elev_mean": "Elevation_Mean",
        "ELEV_MEAN": "Elevation_Mean",
        "Mean": "Elevation_Mean",
        "mean": "Elevation_Mean",
        "mean_elevation": "Elevation_Mean",
        "Elevation": "Elevation_Mean",

        "Elev_median": "Elevation_Median",
        "ELEV_MEDIAN": "Elevation_Median",
        "Median": "Elevation_Median",
        "median": "Elevation_Median",

        "Elev_min": "Elevation_Min",
        "ELEV_MIN": "Elevation_Min",
        "Min": "Elevation_Min",
        "min": "Elevation_Min",

        "Elev_max": "Elevation_Max",
        "ELEV_MAX": "Elevation_Max",
        "Max": "Elevation_Max",
        "max": "Elevation_Max",
    }

    elevation_df = elevation_df.rename(columns={
        col: rename_map[col]
        for col in elevation_df.columns
        if col in rename_map
    })

    return elevation_df


def validate_required_columns(df: pd.DataFrame, required_cols: list, dataset_name: str) -> None:
    """Raise a clear error if required columns are missing."""
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(
            f"{dataset_name} is missing required columns: {missing}. "
            f"Available columns: {list(df.columns)}"
        )

def entropy_weights(dataframe: pd.DataFrame):
    """Entropy weighting across component score columns; zeros handled safely for log."""
    x = dataframe.copy().replace([np.inf, -np.inf], np.nan).fillna(0).clip(lower=0)
    for col in x.columns:
        min_val, max_val = x[col].min(), x[col].max()
        x[col] = (x[col] - min_val) / (max_val - min_val) if max_val != min_val else 0.0

    col_sums = x.sum(axis=0)
    p = x.div(col_sums.replace(0, np.nan), axis=1).fillna(0)
    m = len(x)
    k = 1 / np.log(m) if m > 1 else 1.0
    p_log = p.replace(0, np.nan)
    entropy = (-k * (p * np.log(p_log)).sum(axis=0)).fillna(0)
    diversification = 1 - entropy
    if diversification.sum() == 0:
        weights = pd.Series(1 / len(diversification), index=diversification.index)
    else:
        weights = diversification / diversification.sum()
    return weights, entropy, diversification


def assert_exact_barangay_count(df: pd.DataFrame, stage: str, expected: int = EXPECTED_BARANGAY_COUNT) -> None:
    n = len(df)
    if n != expected:
        raise ValueError(
            f"[{stage}] Expected exactly {expected} barangays, found {n}. "
            "Check merges, filters, and input coverage for City of Manila."
        )
    print(f"[{stage}] Barangay count validated: {n}")


def validation_summary(title: str, checks: dict) -> pd.DataFrame:
    rows = [{"check": k, "count": int(v)} for k, v in checks.items()]
    report = pd.DataFrame(rows)
    print(f"\n=== {title} ===")
    print(report.to_string(index=False))
    return report


print("Helper functions defined.")



Helper functions defined.


## Required QGIS Elevation Data

Elevation is handled as a required tabular input in this notebook.

The barangay-level elevation statistics were generated upstream in QGIS using **Zonal Statistics**. The Manila barangay polygon layer was used as the zone layer, while the DEM/DTM raster was used as the elevation raster.

The expected QGIS output fields are:

- `adm4_name`
- `Elev_count`
- `Elev_mean`
- `Elev_median`
- `Elev_min`
- `Elev_max`

The notebook standardizes these fields as:

- `Barangay`
- `Elevation_Count`
- `Elevation_Mean`
- `Elevation_Median`
- `Elevation_Min`
- `Elevation_Max`

For the Disaster Prioritization Index, `Elevation_Mean` is used as the physical vulnerability indicator. It is reverse-normalized because lower elevation indicates greater flood susceptibility.

The optional DTM extraction helper below is retained only for reproducibility/reference. The final pipeline uses the QGIS-exported elevation CSV.


In [4]:
def extract_barangay_elevation_from_dtm(boundary_path, dtm_path, output_csv):
    """Extract mean/min/max/median elevation per barangay from DTM using zonal statistics."""
    if rasterio is None or zonal_stats is None:
        raise ImportError("Install rasterio and rasterstats to compute elevation from DTM inside the notebook.")

    boundary_path = Path(boundary_path)
    dtm_path = Path(dtm_path)
    output_csv = Path(output_csv)

    if not boundary_path.exists():
        raise FileNotFoundError(f"Barangay boundary file not found: {boundary_path}")
    if not dtm_path.exists():
        raise FileNotFoundError(f"DTM raster not found: {dtm_path}")

    gdf = gpd.read_file(boundary_path)
    gdf.columns = [str(c).strip() for c in gdf.columns]

    # Detect barangay name column
    name_col = next((c for c in ["Barangay", "barangay", "ADM4_EN", "adm4_en", "brgy", "brgy_name", "NAME_4"] if c in gdf.columns), None)
    if name_col is None:
        raise ValueError(f"Cannot detect barangay name column in boundary file. Columns: {list(gdf.columns)}")

    gdf = gdf.copy()
    gdf["Barangay"] = gdf[name_col].astype(str).str.strip()
    if KEEP_ONLY_NUMBERED_BARANGAYS:
        gdf = gdf[gdf["Barangay"].apply(is_numbered_barangay)].copy()

    if gdf.crs is None:
        raise ValueError("Barangay boundary file has no CRS. Define its CRS in QGIS before using it here.")

    # Repair geometries without doing any flood-intersection processing
    gdf["geometry"] = gdf.geometry.buffer(0)
    gdf = gdf[~gdf.geometry.is_empty & gdf.geometry.notna()].copy()

    with rasterio.open(dtm_path) as src:
        raster_crs = src.crs
        nodata = src.nodata

    if raster_crs is None:
        raise ValueError("DTM raster has no CRS. Define/provide a CRS in QGIS first.")

    if gdf.crs != raster_crs:
        gdf = gdf.to_crs(raster_crs)

    stats = zonal_stats(
        gdf,
        str(dtm_path),
        stats=["mean", "min", "max", "median"],
        nodata=nodata,
        all_touched=True
    )

    elev = pd.DataFrame(stats).rename(columns={
        "mean": "Elevation_Mean",
        "min": "Elevation_Min",
        "max": "Elevation_Max",
        "median": "Elevation_Median",
    })
    elev.insert(0, "Barangay", gdf["Barangay"].values)
    elev["barangay_key_strict"] = elev["Barangay"].apply(clean_barangay_name_strict)
    elev.to_csv(output_csv, index=False)
    print(f"Elevation CSV generated from DTM: {output_csv}")
    print("Rows:", len(elev), "Missing mean elevation:", int(elev["Elevation_Mean"].isna().sum()))
    return elev


## 4. Load QGIS-Prepared Flood CSV

**Purpose:** Load the validated barangay flood-area table that was already produced separately in QGIS.

This notebook intentionally **does not** load shapefiles, repair geometries, reproject layers, intersect flood polygons, or convert shapefiles to CSV. Those spatial operations are treated as completed upstream.

In [5]:
def standardize_flood_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Standardize common QGIS/export column variants to the names used downstream."""
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    lower_map = {c: str(c).lower().strip().replace(" ", "_") for c in df.columns}
    df = df.rename(columns=lower_map)

    rename_candidates = {
        "barangay": ["barangay", "brgy", "brgy_name", "barangay_name", "adm4_name"],
        "brgy_area_sqm": ["brgy_area_sqm", "barangay_area_sqm", "area_sqm", "total_area_sqm", "shape_area"],
        "flood_area_5yr_sqm": ["flood_area_5yr_sqm", "flood_5yr_sqm", "flood_area_5yr", "area_5yr_sqm"],
        "flood_pct_5yr": ["flood_pct_5yr", "flood_5yr_pct", "pct_5yr", "flood_percent_5yr"],
        "flood_area_25yr_sqm": ["flood_area_25yr_sqm", "flood_25yr_sqm", "flood_area_25yr", "area_25yr_sqm"],
        "flood_pct_25yr": ["flood_pct_25yr", "flood_25yr_pct", "pct_25yr", "flood_percent_25yr"],
        "flood_area_100yr_sqm": ["flood_area_100yr_sqm", "flood_100yr_sqm", "flood_area_100yr", "area_100yr_sqm"],
        "flood_pct_100yr": ["flood_pct_100yr", "flood_100yr_pct", "pct_100yr", "flood_percent_100yr"],
    }

    for target, candidates in rename_candidates.items():
        if target not in df.columns:
            match = next((c for c in candidates if c in df.columns), None)
            if match:
                df = df.rename(columns={match: target})

    required = [
        "barangay", "brgy_area_sqm",
        "flood_area_5yr_sqm", "flood_pct_5yr",
        "flood_area_25yr_sqm", "flood_pct_25yr",
    ]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Flood CSV is missing required columns: {missing}. Available columns: {list(df.columns)}")

    df = safe_numeric(df, [c for c in df.columns if any(k in c for k in ["area", "pct", "sqm"])] )
    df["barangay"] = df["barangay"].astype(str).str.strip()
    df["barangay_no"] = df["barangay"].apply(extract_barangay_number)

    if KEEP_ONLY_NUMBERED_BARANGAYS:
        before = len(df)
        df = df[df["barangay"].apply(is_numbered_barangay)].copy()
        print(f"Numbered barangay filter: {before} -> {len(df)}")

    df = df.drop_duplicates(subset=["barangay"]).copy()
    df["Return_Period_Valid_5_25"] = df["flood_pct_25yr"] >= df["flood_pct_5yr"]
    df["flag_25yr_less_than_5yr"] = ~df["Return_Period_Valid_5_25"]
    return df

if not FLOOD_CSV.exists():
    raise FileNotFoundError(f"Required QGIS-prepared flood CSV not found: {FLOOD_CSV}")

flood_spatial_df = standardize_flood_columns(pd.read_csv(FLOOD_CSV))
assert_exact_barangay_count(flood_spatial_df, stage="qgis_flood_csv_load")

print("Loaded QGIS-prepared flood CSV:", FLOOD_CSV)
print("Rows:", len(flood_spatial_df))
print("5yr/25yr return-period anomalies:", int(flood_spatial_df["flag_25yr_less_than_5yr"].sum()))
display(flood_spatial_df.head())


Numbered barangay filter: 897 -> 897
[qgis_flood_csv_load] Barangay count validated: 897
Loaded QGIS-prepared flood CSV: MetroManila_Barangay_Flood_Area_Validated.csv
Rows: 897
5yr/25yr return-period anomalies: 2


,barangay,brgy_area_sqm,flood_area_5yr_sqm,flood_pct_5yr,flood_area_25yr_sqm,flood_pct_25yr,flood_area_100yr_sqm,flood_pct_100yr,flag_25yr_less_than_5yr,flag_100yr_less_than_25yr,flag_100yr_less_than_5yr,barangay_no,Return_Period_Valid_5_25
0,Barangay 1,49547.490522,8264.845530,16.680654,21069.565046,42.523980,24194.317057,48.830560,False,False,False,1,True
1,Barangay 10,6057.586255,5544.134717,91.523826,5999.129222,99.034978,6033.964192,99.610042,False,False,False,10,True
2,Barangay 100,29349.500168,8985.706920,30.616218,14348.687691,48.889036,15223.078876,51.868273,False,False,False,100,True
3,Barangay 101,103997.369545,37525.242932,36.082877,57378.997256,55.173508,61606.839736,59.238844,False,False,False,101,True
4,Barangay 102,23899.675836,4506.429582,18.855610,14200.178503,59.415779,20019.915620,83.766473,False,False,False,102,True


## 5. Validate and Copy Prepared Flood CSV

**Purpose:** Validate the already-prepared flood CSV and copy it to the output directory for reproducibility. No shapefile conversion is performed.

In [6]:
export_cols = [
    "barangay", "brgy_area_sqm",
    "flood_area_5yr_sqm", "flood_pct_5yr",
    "flood_area_25yr_sqm", "flood_pct_25yr",
    "Return_Period_Valid_5_25", "flag_25yr_less_than_5yr", "barangay_no",
]
export_cols = [c for c in export_cols if c in flood_spatial_df.columns]
flood_spatial_df[export_cols].to_csv(OUT_FLOOD_CSV, index=False)
print(f"Copied validated QGIS flood CSV to: {OUT_FLOOD_CSV}")
print("Rows:", len(flood_spatial_df))
display(flood_spatial_df[export_cols].head())


Copied validated QGIS flood CSV to: outputs/MetroManila_Barangay_Flood_Area_Validated.csv
Rows: 897


,barangay,brgy_area_sqm,flood_area_5yr_sqm,flood_pct_5yr,flood_area_25yr_sqm,flood_pct_25yr,Return_Period_Valid_5_25,flag_25yr_less_than_5yr,barangay_no
0,Barangay 1,49547.490522,8264.845530,16.680654,21069.565046,42.523980,True,False,1
1,Barangay 10,6057.586255,5544.134717,91.523826,5999.129222,99.034978,True,False,10
2,Barangay 100,29349.500168,8985.706920,30.616218,14348.687691,48.889036,True,False,100
3,Barangay 101,103997.369545,37525.242932,36.082877,57378.997256,55.173508,True,False,101
4,Barangay 102,23899.675836,4506.429582,18.855610,14200.178503,59.415779,True,False,102


## 6. Loading of PSA 2020 and PSA 2024 population datasets

**Purpose:** Load official PSA population snapshots for both census/reference years used in the thesis.

**Input:**
- PSA 2024: `population-dataset.csv`
- PSA 2020: `NCR.xlsx` if present; otherwise `bgypop2020` from `manila_final_base_dataset_NEW.csv`

**Output:** Clean population tables with `Population_2020` and `Population_2024`.

**Why both years are integrated:** Population Change 2020–2024 (%) requires both snapshots. Using only one year would omit the demographic-change vulnerability indicator required by the updated methodology.


In [9]:
def load_psa_2024(csv_path):
    """
    Load PSA 2024 barangay population data.

    Supports:
    1. Headerless two-column CSV:
       Barangay, Population
    2. CSV with named barangay/population columns.
    """

    csv_path = Path(csv_path)

    if not csv_path.exists():
        raise FileNotFoundError(f"PSA 2024 file not found: {csv_path}")

    # Try headered read first
    try:
        df_raw = pd.read_csv(csv_path)
        df_raw.columns = [str(c).strip() for c in df_raw.columns]
    except Exception:
        df_raw = pd.DataFrame()

    barangay_candidates = [
        "Barangay", "barangay", "Barangay Name", "brgy_name",
        "adm4_name", "ADM4_NAME"
    ]

    pop_candidates = [
        "Population_2024", "population_2024", "Pop_2024",
        "Population", "population", "PSA_2024"
    ]

    barangay_col = next((c for c in barangay_candidates if c in df_raw.columns), None)
    pop_col = next((c for c in pop_candidates if c in df_raw.columns), None)

    # If headered detection fails, read as headerless two-column CSV
    if barangay_col is None or pop_col is None:
        df_raw = pd.read_csv(csv_path, header=None)
        df_raw = df_raw.dropna(how="all").dropna(axis=1, how="all")

        if df_raw.shape[1] < 2:
            raise ValueError(f"Could not detect barangay/population columns in {csv_path}")

        df = df_raw.iloc[:, :2].copy()
        df.columns = ["Barangay", "Population_2024"]

    else:
        df = df_raw[[barangay_col, pop_col]].copy()
        df.columns = ["Barangay", "Population_2024"]

    df["Barangay"] = df["Barangay"].astype(str).str.strip()

    df["Population_2024"] = (
        df["Population_2024"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.strip()
    )
    df["Population_2024"] = pd.to_numeric(df["Population_2024"], errors="coerce")

    # Keep valid barangay rows only
    df = df[
        df["Barangay"].str.contains(r"Barangay\s+\d+", case=False, na=False)
    ].copy()

    df = df[df["Population_2024"].notna()].copy()

    df["barangay_key_strict"] = df["Barangay"].apply(clean_barangay_name_strict)

    # Remove duplicate barangay records if present
    df = df.drop_duplicates("barangay_key_strict", keep="first").copy()

    # Sort
    df["Barangay_No"] = df["Barangay"].apply(extract_number)
    df = df.sort_values(["Barangay_No", "Barangay"]).drop(columns=["Barangay_No"]).reset_index(drop=True)

    print(f"Loaded PSA 2024 population: {len(df)} barangays")
    print("PSA 2024 total population:", int(df["Population_2024"].sum()))

    return df[["Barangay", "Population_2024", "barangay_key_strict"]]

def load_psa_2020(xlsx_path, base_csv=None):
    """
    Load PSA 2020 barangay population data.

    Supports:
    1. Simple two-column Excel layout:
       Barangay | Population
    2. Headerless two-column Excel layout.
    3. Multi-sheet PSA-style files with barangay and population columns.
    4. Optional fallback to base admin CSV if it contains 2020 population.
    """

    xlsx_path = Path(xlsx_path)

    if not xlsx_path.exists():
        raise FileNotFoundError(f"PSA 2020 file not found: {xlsx_path}")

    frames = []

    # Read all sheets with no assumed header first
    sheets = pd.read_excel(xlsx_path, sheet_name=None, header=None)

    for sheet_name, raw in sheets.items():
        tmp = raw.copy()
        tmp = tmp.dropna(how="all").dropna(axis=1, how="all")

        if tmp.empty or tmp.shape[1] < 2:
            continue

        # Flatten to first two useful columns
        tmp = tmp.iloc[:, :2].copy()
        tmp.columns = ["Barangay", "Population_2020"]

        tmp["Barangay"] = tmp["Barangay"].astype(str).str.strip()
        tmp["Population_2020"] = (
            tmp["Population_2020"]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.strip()
        )
        tmp["Population_2020"] = pd.to_numeric(tmp["Population_2020"], errors="coerce")

        # Keep rows that look like Manila barangay rows
        tmp = tmp[
            tmp["Barangay"].str.contains(r"Barangay\s+\d+", case=False, na=False)
        ].copy()

        tmp = tmp[tmp["Population_2020"].notna()].copy()

        if len(tmp) > 0:
            frames.append(tmp)

    if not frames:
        # Optional fallback to base admin file
        if base_csv is not None and Path(base_csv).exists():
            base = pd.read_csv(base_csv)
            base.columns = [str(c).strip() for c in base.columns]

            brgy_col = next(
                (
                    c for c in [
                        "Barangay", "barangay", "adm4_name", "ADM4_NAME",
                        "brgy_name", "Brgy_Name", "NAME_4"
                    ]
                    if c in base.columns
                ),
                None
            )

            pop_col = next(
                (
                    c for c in [
                        "Population_2020", "Pop_2020", "bgypop2020",
                        "bgy_pop2020", "pop2020", "PSA_2020"
                    ]
                    if c in base.columns
                ),
                None
            )

            if brgy_col and pop_col:
                df = base[[brgy_col, pop_col]].copy()
                df.columns = ["Barangay", "Population_2020"]
                df["Population_2020"] = (
                    df["Population_2020"]
                    .astype(str)
                    .str.replace(",", "", regex=False)
                    .str.strip()
                )
                df["Population_2020"] = pd.to_numeric(df["Population_2020"], errors="coerce")
                df = df[df["Population_2020"].notna()].copy()
                df["barangay_key_strict"] = df["Barangay"].apply(clean_barangay_name_strict)
                df = df.drop_duplicates("barangay_key_strict")
                print(f"Loaded PSA 2020 fallback from base admin CSV: {len(df)} rows")
                return df[["Barangay", "Population_2020", "barangay_key_strict"]]

        raise ValueError(f"Could not detect barangay/population columns in {xlsx_path}")

    df = pd.concat(frames, ignore_index=True)

    df["Barangay"] = df["Barangay"].astype(str).str.strip()
    df["barangay_key_strict"] = df["Barangay"].apply(clean_barangay_name_strict)

    # Remove duplicate barangay records if any
    df = df.drop_duplicates("barangay_key_strict", keep="first").copy()

    # Final sort
    df["Barangay_No"] = df["Barangay"].apply(extract_number)
    df = df.sort_values(["Barangay_No", "Barangay"]).drop(columns=["Barangay_No"]).reset_index(drop=True)

    print(f"Loaded PSA 2020 population: {len(df)} barangays")
    print("PSA 2020 total population:", int(df["Population_2020"].sum()))

    return df[["Barangay", "Population_2020", "barangay_key_strict"]]


df_pop_2024 = load_psa_2024(PSA_2024_CSV)
df_pop_2020 = load_psa_2020(PSA_2020_XLSX, BASE_ADMIN_CSV)


Loaded PSA 2024 population: 897 barangays
PSA 2024 total population: 1902590
Loaded PSA 2020 population: 897 barangays
PSA 2020 total population: 1846513


## 7. Barangay name/key standardization

**Purpose:** Create consistent merge keys across flood, population, and administrative tables.

**Input:** Raw barangay name strings from each source.

**Output:** `barangay_clean` and `barangay_key_strict` fields.

**Why needed:** Sources use inconsistent spacing, casing, and hyphenation (e.g., `Brgy. 202-A` vs `Barangay 202A`). Standardization is required for a complete 897-barangay match.


In [10]:
def add_merge_keys(df: pd.DataFrame, barangay_col: str = "Barangay") -> pd.DataFrame:
    df = df.copy()
    df[barangay_col] = df[barangay_col].astype(str).str.strip()
    df["barangay_clean"] = df[barangay_col].apply(clean_barangay_name)
    df["barangay_key_strict"] = df[barangay_col].apply(clean_barangay_name_strict)
    return df


df_pop_2024 = add_merge_keys(df_pop_2024, "Barangay")
df_pop_2020 = add_merge_keys(df_pop_2020, "Barangay")

df_flood = flood_spatial_df.rename(columns={
    "barangay": "Barangay",
    "brgy_area_sqm": "Barangay_Area_Sqm",
    "flood_area_5yr_sqm": "Flood_Area_5yr_Sqm",
    "flood_pct_5yr": "Flood_PCT_5yr",
    "flood_area_25yr_sqm": "Flood_Area_25yr_Sqm",
    "flood_pct_25yr": "Flood_PCT_25yr",
    "barangay_no": "Barangay_No",
}).copy()
df_flood = add_merge_keys(df_flood, "Barangay")

print("Merge keys created for population and flood tables.")
print("2024 keys sample:", df_pop_2024[["Barangay", "barangay_key_strict"]].head(3).to_dict("records"))
print("Flood keys sample:", df_flood[["Barangay", "barangay_key_strict"]].head(3).to_dict("records"))


Merge keys created for population and flood tables.
2024 keys sample: [{'Barangay': 'Barangay 1', 'barangay_key_strict': 'BARANGAY1'}, {'Barangay': 'Barangay 2', 'barangay_key_strict': 'BARANGAY2'}, {'Barangay': 'Barangay 3', 'barangay_key_strict': 'BARANGAY3'}]
Flood keys sample: [{'Barangay': 'Barangay 1', 'barangay_key_strict': 'BARANGAY1'}, {'Barangay': 'Barangay 10', 'barangay_key_strict': 'BARANGAY10'}, {'Barangay': 'Barangay 100', 'barangay_key_strict': 'BARANGAY100'}]


## 8. Population dataset validation and 897-barangay matching

**Purpose:** Merge PSA 2020 and PSA 2024 onto the flood barangay universe and validate complete coverage.

**Input:** Flood table (897 barangays) + PSA 2020 + PSA 2024.

**Output:** Matched barangay table with both population years; citywide total checks.

**Why needed:** The thesis uses total enumeration of Manila's 897 barangays. Missing population values would bias exposure and vulnerability scores.


In [11]:
df = df_flood.merge(
    df_pop_2024[["barangay_key_strict", "Population_2024"]],
    on="barangay_key_strict",
    how="left",
)
df = df.merge(
    df_pop_2020[["barangay_key_strict", "Population_2020"]],
    on="barangay_key_strict",
    how="left",
)

print("After population merges:", len(df))
print("Missing Population_2024:", int(df["Population_2024"].isna().sum()))
print("Missing Population_2020:", int(df["Population_2020"].isna().sum()))

missing_2024 = df.loc[df["Population_2024"].isna(), ["Barangay", "barangay_key_strict"]]
missing_2020 = df.loc[df["Population_2020"].isna(), ["Barangay", "barangay_key_strict"]]
if len(missing_2024):
    print("Barangays missing PSA 2024:")
    print(missing_2024.head(20).to_string(index=False))
if len(missing_2020):
    print("Barangays missing PSA 2020:")
    print(missing_2020.head(20).to_string(index=False))

if df["Population_2024"].isna().any() or df["Population_2020"].isna().any():
    raise ValueError(
        "Population integration incomplete: one or more barangays lack Population_2020 or Population_2024."
    )

assert_exact_barangay_count(df, stage="population_match")

total_2020 = float(df["Population_2020"].sum())
total_2024 = float(df["Population_2024"].sum())
change_abs = total_2024 - total_2020
change_pct = (change_abs / total_2020) * 100 if total_2020 else np.nan

print(f"\nCitywide Population_2020: {total_2020:,.0f} (expected {EXPECTED_PSA_2020_TOTAL:,})")
print(f"Citywide Population_2024: {total_2024:,.0f} (expected {EXPECTED_PSA_2024_TOTAL:,})")
print(f"Absolute change: {change_abs:,.0f} (expected {EXPECTED_POP_CHANGE_ABS:,})")
print(f"Percentage change: {change_pct:.4f}% (expected ~{EXPECTED_POP_CHANGE_PCT}%)")

if abs(total_2020 - EXPECTED_PSA_2020_TOTAL) > 1:
    print("WARNING: PSA 2020 citywide total differs from expected reference.")
if abs(total_2024 - EXPECTED_PSA_2024_TOTAL) > 1:
    print("WARNING: PSA 2024 citywide total differs from expected reference.")
if abs(change_abs - EXPECTED_POP_CHANGE_ABS) > 1:
    print("WARNING: Absolute population change differs from expected reference.")


After population merges: 897
Missing Population_2024: 0
Missing Population_2020: 0
[population_match] Barangay count validated: 897

Citywide Population_2020: 1,846,513 (expected 1,846,513)
Citywide Population_2024: 1,902,590 (expected 1,902,590)
Absolute change: 56,077 (expected 56,077)
Percentage change: 3.0369% (expected ~3.04%)


## 9. Integration of flood, population, administrative, area, and required elevation data

**Purpose:** Attach administrative identifiers, official area metadata, and QGIS-derived elevation statistics.

**Input:** Matched flood-population table + `manila_final_base_dataset_NEW.csv` + required QGIS Zonal Statistics elevation CSV.

**Output:** Integrated analytical dataframe for feature engineering.

**Why needed:** District context features and elevation susceptibility require administrative joins beyond flood and population alone.

**Elevation rule:** `Elevation_Mean` must be present for all 897 barangays. If any barangay lacks mean elevation after the merge, the notebook raises an error.


In [12]:
# ============================================================
# Merge required QGIS elevation data
# ============================================================

if not ELEVATION_CSV.exists():
    raise FileNotFoundError(
        f"Required elevation CSV not found: {ELEVATION_CSV}. "
        "Provide the QGIS Zonal Statistics output file."
    )

# Load and standardize QGIS elevation table
elevation_df = pd.read_csv(ELEVATION_CSV)
elevation_df = standardize_elevation_columns(elevation_df)

validate_required_columns(
    elevation_df,
    ["Barangay", "Elevation_Mean"],
    dataset_name="Elevation CSV"
)

# Build strict merge key
elevation_df["barangay_key_strict"] = elevation_df["Barangay"].apply(clean_barangay_name_strict)

# Keep only useful elevation columns that exist
elevation_keep_cols = [
    "barangay_key_strict",
    "Barangay",
    "Elevation_Count",
    "Elevation_Mean",
    "Elevation_Median",
    "Elevation_Min",
    "Elevation_Max",
]
elevation_keep_cols = [c for c in elevation_keep_cols if c in elevation_df.columns]
elevation_df = elevation_df[elevation_keep_cols].copy()

# Convert elevation values to numeric
numeric_elev_cols = [
    "Elevation_Count",
    "Elevation_Mean",
    "Elevation_Median",
    "Elevation_Min",
    "Elevation_Max",
]
elevation_df = safe_numeric(elevation_df, numeric_elev_cols)

# Check duplicate barangay keys in elevation source
duplicate_elev = elevation_df[elevation_df["barangay_key_strict"].duplicated(keep=False)]
if len(duplicate_elev) > 0:
    print("Duplicate barangay keys found in elevation file:")
    display(duplicate_elev.sort_values("barangay_key_strict"))
    raise ValueError("Duplicate barangay keys found in elevation data. Clean the elevation CSV before merging.")

# Optional diagnostic: show non-standard/non-numbered entries
non_numbered_elev = elevation_df[
    ~elevation_df["Barangay"].astype(str).apply(is_numbered_barangay)
].copy()

if len(non_numbered_elev) > 0:
    print("Non-numbered rows detected in elevation file. These will only merge if present in the main dataset:")
    display(non_numbered_elev[["Barangay"]].head(20))

# Merge elevation into main dataframe
df = df.merge(
    elevation_df.drop(columns=["Barangay"], errors="ignore"),
    on="barangay_key_strict",
    how="left"
)

print("Elevation merged from:", ELEVATION_CSV)
print("Dataset shape after elevation merge:", df.shape)
print("Missing Elevation_Mean:", int(df["Elevation_Mean"].isna().sum()))

# Validate barangay count remains 897
assert_exact_barangay_count(df, stage="after_elevation_merge")

# Elevation is required for final methodology
missing_elev_count = int(df["Elevation_Mean"].isna().sum())
if missing_elev_count > 0:
    missing_elev = df.loc[
        df["Elevation_Mean"].isna(),
        ["Barangay", "barangay_key_strict"]
    ].copy()

    print("Barangays missing elevation:")
    display(missing_elev)

    raise ValueError(
        f"{missing_elev_count} barangays are missing Elevation_Mean after merge. "
        "Check barangay naming/key matching between the main dataset and the QGIS elevation CSV."
    )

print("Elevation validation passed: all 897 barangays have Elevation_Mean.")

# Quick elevation summary for audit
display(
    df[["Elevation_Mean", "Elevation_Median", "Elevation_Min", "Elevation_Max"]]
    .describe()
    .T
)

Elevation merged from: manila_barangay_elevation.csv
Dataset shape after elevation merge: (897, 22)
Missing Elevation_Mean: 0
[after_elevation_merge] Barangay count validated: 897
Elevation validation passed: all 897 barangays have Elevation_Mean.


,count,mean,std,min,25%,50%,75%,max
Elevation_Mean,897.0,2.328222,1.200392,0.0,1.691874,2.0,3.0,9.141101
Elevation_Median,897.0,2.347269,1.192927,0.0,2.000000,2.0,3.0,9.000000
Elevation_Min,897.0,1.746934,1.180928,0.0,1.000000,2.0,2.0,8.000000
Elevation_Max,897.0,2.926421,1.850779,0.0,2.000000,3.0,3.0,30.000000


## 14. Feature engineering

**Purpose:** Compute thesis indicators for hazard escalation, exposure, density, population change, and district context.

**Key definitions (updated methodology):**
- `Population_Change_2020_2024 = Population_2024 - Population_2020`
- `Population_Change_2020_2024_Pct = ((Population_2024 - Population_2020) / Population_2020) * 100` — **not annualized**
- `Barangay_Area_Hectares = Barangay_Area_Sqm / 10000`
- `Population_Density_per_Hectare = Population_2024 / Barangay_Area_Hectares`
- Affected population uses **PSA 2024** only
- `Flood_Escalation_5to25` and `Flood_Severity_Increase_5to25_Sqm` flagged if negative

**Why Population Change (not growth rate):** The indicator is a four-year observed change between official PSA snapshots, not an annualized demographic rate.

**Why density uses 2024:** Exposure and current vulnerability should reflect the more recent population snapshot.

**Why Relative Population Density is removed:** The updated vulnerability specification uses absolute density, population change, and elevation only. Relative density is excluded from vulnerability and from the final ML feature list.


In [13]:
df = safe_numeric(
    df,
    [
        "Population_2020", "Population_2024", "Barangay_Area_Sqm",
        "Flood_Area_5yr_Sqm", "Flood_PCT_5yr", "Flood_Area_25yr_Sqm", "Flood_PCT_25yr",
        "bgyarea_sqkm", "Elevation_Mean", "Elevation_Median", "Elevation_Min", "Elevation_Max", "Elevation_Count",
    ],
)

df["Population_Change_2020_2024"] = df["Population_2024"] - df["Population_2020"]
df["Population_Change_2020_2024_Pct"] = np.where(
    df["Population_2020"] > 0,
    (df["Population_Change_2020_2024"] / df["Population_2020"]) * 100,
    np.nan,
)

df["Barangay_Area_Hectares"] = df["Barangay_Area_Sqm"] / 10_000
if (df["Barangay_Area_Hectares"] <= 0).any():
    raise ValueError("Non-positive barangay area in hectares detected.")
df["Population_Density_per_Hectare"] = df["Population_2024"] / df["Barangay_Area_Hectares"]

df["Affected_Population_5yr"] = df["Population_2024"] * (df["Flood_PCT_5yr"] / 100)
df["Affected_Population_25yr"] = df["Population_2024"] * (df["Flood_PCT_25yr"] / 100)
df["Affected_Population_Increase_5to25"] = (
    df["Affected_Population_25yr"] - df["Affected_Population_5yr"]
)

df["Flood_Escalation_5to25"] = df["Flood_PCT_25yr"] - df["Flood_PCT_5yr"]
df["Flood_Severity_Increase_5to25_Sqm"] = df["Flood_Area_25yr_Sqm"] - df["Flood_Area_5yr_Sqm"]

neg_esc = df["Flood_Escalation_5to25"] < -EPS
neg_sev = df["Flood_Severity_Increase_5to25_Sqm"] < -EPS
if neg_esc.any() or neg_sev.any():
    print(
        f"WARNING: Negative escalation/severity detected "
        f"(escalation={int(neg_esc.sum())}, severity_sqm={int(neg_sev.sum())})."
    )
    print(
        df.loc[
            neg_esc | neg_sev,
            [
                "Barangay", "Flood_PCT_5yr", "Flood_PCT_25yr",
                "Flood_Escalation_5to25", "Flood_Severity_Increase_5to25_Sqm",
            ],
        ].head(20)
    )

if "bgyarea_sqkm" in df.columns:
    df["Base_Area_Sqm"] = df["bgyarea_sqkm"] * 1_000_000
    df["Area_Difference_Sqm"] = df["Barangay_Area_Sqm"] - df["Base_Area_Sqm"]
    df["Area_Difference_Pct"] = np.where(
        df["Base_Area_Sqm"] > 0,
        (df["Area_Difference_Sqm"] / df["Base_Area_Sqm"]) * 100,
        np.nan,
    )

if "city_name" in df.columns:
    df["District_Avg_Flood_PCT_5yr"] = df.groupby("city_name")["Flood_PCT_5yr"].transform("mean")
    df["District_Avg_Flood_PCT_25yr"] = df.groupby("city_name")["Flood_PCT_25yr"].transform("mean")
    df["District_Avg_Population_Density_per_Hectare"] = (
        df.groupby("city_name")["Population_Density_per_Hectare"].transform("mean")
    )
    df["Relative_Flood_Risk_25yr_vs_District"] = (
        df["Flood_PCT_25yr"] - df["District_Avg_Flood_PCT_25yr"]
    )
    # NOTE: Relative Population Density is intentionally NOT computed for vulnerability/ML.

df["Sort_Key"] = df["Barangay"].apply(extract_number)
df = df.sort_values(["Sort_Key", "Barangay"]).drop(columns=["Sort_Key"]).reset_index(drop=True)

print("Feature engineering complete.")
print(
    df[[
        "Barangay", "Population_2020", "Population_2024", "Population_Change_2020_2024_Pct",
        "Population_Density_per_Hectare", "Flood_Escalation_5to25",
        "Affected_Population_5yr", "Affected_Population_25yr",
        "Elevation_Mean", "Elevation_Median",
    ]].head()
)



Feature engineering complete.
     Barangay  Population_2020  Population_2024  Population_Change_2020_2024_Pct  Population_Density_per_Hectare  Flood_Escalation_5to25  \
0  Barangay 1             2995           3119.0                         4.140234                      629.497068               25.843326   
1  Barangay 2             1887           1666.0                       -11.711712                      439.740638               30.180490   
2  Barangay 3             1341           1232.0                        -8.128262                      316.379617               25.033508   
3  Barangay 4             1691           1717.0                         1.537552                      427.181509               43.834370   
4  Barangay 5             1288           1321.0                         2.562112                      533.913235               43.535626   

   Affected_Population_5yr  Affected_Population_25yr  Elevation_Mean  Elevation_Median  
0               520.269603              

## 15. Export of enriched analytical dataset

**Purpose:** Save the full integrated barangay table for documentation and downstream DPI/ML.

**Input:** Feature-engineered dataframe.

**Output:**
- `manila-city-flood-population_enriched_NEW.csv`
- `manila-city-flood-population_enriched_NEW.xlsx`

**Why needed:** Provides a human-readable and machine-readable archive of all analytical fields before index construction.


In [14]:
assert_exact_barangay_count(df, stage="enriched_export")

geom_cols = [c for c in df.columns if str(c).lower() in {"geometry", "geom"}]
if geom_cols:
    df = df.drop(columns=geom_cols)

df.to_csv(OUT_ENRICHED_CSV, index=False)
try:
    df.to_excel(OUT_ENRICHED_XLSX, index=False)
    print(f"Exported Excel: {OUT_ENRICHED_XLSX}")
except Exception as exc:
    print(f"Excel export skipped ({exc})")

print(f"Exported enriched CSV: {OUT_ENRICHED_CSV}")
print("Final enriched shape:", df.shape)
print("Missing value summary (top 15):")
print(df.isna().sum().sort_values(ascending=False).head(15))


[enriched_export] Barangay count validated: 897
Exported Excel: outputs/manila-city-flood-population_enriched_NEW.xlsx
Exported enriched CSV: outputs/manila-city-flood-population_enriched_NEW.csv
Final enriched shape: (897, 31)
Missing value summary (top 15):
Barangay                     0
Barangay_Area_Sqm            0
Flood_Area_5yr_Sqm           0
Flood_PCT_5yr                0
Flood_Area_25yr_Sqm          0
Flood_PCT_25yr               0
flood_area_100yr_sqm         0
flood_pct_100yr              0
flag_25yr_less_than_5yr      0
flag_100yr_less_than_25yr    0
flag_100yr_less_than_5yr     0
Barangay_No                  0
Return_Period_Valid_5_25     0
barangay_clean               0
barangay_key_strict          0
dtype: int64


## 16. Preparation of ML-ready dataset scaffold

**Purpose:** Identify identifier columns versus candidate analytic fields ahead of CSI/DPI computation.

**Input:** Enriched dataset.

**Output:** Working dataframe `df_ml` for index and modeling stages.

**Why needed:** Separates identifiers from numeric indicators and prepares a clean table for leakage-controlled modeling.


In [15]:
df_ml = df.copy()
df_ml.columns = df_ml.columns.astype(str).str.strip()

id_cols = keep_existing(
    [
        "Barangay", "Barangay_No", "psgc_10d", "city_name", "brgy_code",
        "Population_2020", "Population_2024",
    ],
    df_ml,
)
print("Identifier / reference columns:", id_cols)
print("Working ML dataframe shape:", df_ml.shape)
assert_exact_barangay_count(df_ml, stage="ml_scaffold")


Identifier / reference columns: ['Barangay', 'Barangay_No', 'Population_2020', 'Population_2024']
Working ML dataframe shape: (897, 31)
[ml_scaffold] Barangay count validated: 897


## 17. CSI computation

**Purpose:** Compute the Composite Susceptibility Index (hazard component) from validated 5-year and 25-year indicators only.

**Indicators (equal-weighted after min-max normalization):**
- `Flood_PCT_5yr`
- `Flood_PCT_25yr`
- `Flood_Escalation_5to25`
- `Flood_Severity_Increase_5to25_Sqm`

`CSI = mean(normalized CSI indicators)`

**Why CSI is separate:** DPI follows a multi-criteria structure where hazard, exposure, and vulnerability are computed independently before entropy weighting. Separating CSI keeps the hazard construct interpretable.

**Why the 100-year layer is excluded:** Return-period consistency issues in the 100-year source layer would distort CSI and downstream DPI classes.


In [16]:
csi_inputs = [
    "Flood_PCT_5yr",
    "Flood_PCT_25yr",
    "Flood_Escalation_5to25",
    "Flood_Severity_Increase_5to25_Sqm",
]
missing = [c for c in csi_inputs if c not in df_ml.columns]
if missing:
    raise ValueError(f"Missing CSI inputs: {missing}")

df_ml = safe_numeric(df_ml, csi_inputs)
df_ml["norm_Flood_PCT_5yr"] = minmax_normalize(df_ml["Flood_PCT_5yr"])
df_ml["norm_Flood_PCT_25yr"] = minmax_normalize(df_ml["Flood_PCT_25yr"])
df_ml["norm_Flood_Escalation_5to25"] = minmax_normalize(df_ml["Flood_Escalation_5to25"])
df_ml["norm_Flood_Severity_Increase_5to25_Sqm"] = minmax_normalize(
    df_ml["Flood_Severity_Increase_5to25_Sqm"]
)

df_ml["CSI"] = df_ml[[
    "norm_Flood_PCT_5yr",
    "norm_Flood_PCT_25yr",
    "norm_Flood_Escalation_5to25",
    "norm_Flood_Severity_Increase_5to25_Sqm",
]].mean(axis=1)

print("CSI computed. Summary:")
print(df_ml["CSI"].describe())


CSI computed. Summary:
count    8.970000e+02
mean     4.205806e-01
std      1.474927e-01
min      1.754438e-12
25%      3.562610e-01
50%      5.000000e-01
75%      5.105478e-01
max      6.404339e-01
Name: CSI, dtype: float64


## 18. Exposure score computation

**Purpose:** Quantify population potentially affected under 5-year and 25-year flood scenarios using PSA 2024.

**Indicators (equal-weighted after normalization):**
- `Affected_Population_5yr`
- `Affected_Population_25yr`
- `Affected_Population_Increase_5to25`

`Exposure = mean(normalized exposure indicators)`

**Why separate from CSI:** Exposure captures who is affected, not only how much land is flooded.


In [17]:
exposure_inputs = [
    "Affected_Population_5yr",
    "Affected_Population_25yr",
    "Affected_Population_Increase_5to25",
]
missing = [c for c in exposure_inputs if c not in df_ml.columns]
if missing:
    raise ValueError(f"Missing Exposure inputs: {missing}")

df_ml = safe_numeric(df_ml, exposure_inputs)
df_ml["norm_Affected_Population_5yr"] = minmax_normalize(df_ml["Affected_Population_5yr"])
df_ml["norm_Affected_Population_25yr"] = minmax_normalize(df_ml["Affected_Population_25yr"])
df_ml["norm_Affected_Population_Increase_5to25"] = minmax_normalize(
    df_ml["Affected_Population_Increase_5to25"]
)
df_ml["Exposure"] = df_ml[[
    "norm_Affected_Population_5yr",
    "norm_Affected_Population_25yr",
    "norm_Affected_Population_Increase_5to25",
]].mean(axis=1)

print("Exposure computed. Summary:")
print(df_ml["Exposure"].describe())


Exposure computed. Summary:
count    8.970000e+02
mean     8.724946e-02
std      8.365317e-02
min      5.524718e-13
25%      3.956010e-02
50%      6.894792e-02
75%      1.065094e-01
max      8.478075e-01
Name: Exposure, dtype: float64


## 19. Vulnerability score computation

**Purpose:** Compute demographic/physical susceptibility using the final three-indicator specification.

**Indicators:**
1. `Population_Density_per_Hectare` — min-max normalized
2. `Population_Change_2020_2024_Pct` — min-max normalized
3. `Elevation_Mean` — reverse min-max normalized

The vulnerability score is:

$$Vulnerability_i = \frac{NormDensity_i + NormPopChange_i + ReverseNormElevation_i}{3}$$

**Why elevation is reverse-normalized:** Lower-lying barangays are generally more susceptible to inundation, so lower mean elevation receives a higher vulnerability contribution.

**Final methodology rule:** Elevation is required. The previous two-indicator fallback is removed for final thesis runs.


In [18]:
# ============================================================
# Vulnerability score computation
# ============================================================

vuln_inputs = [
    "Population_Density_per_Hectare",
    "Population_Change_2020_2024_Pct",
    "Elevation_Mean",
]

missing = [c for c in vuln_inputs if c not in df_ml.columns]
if missing:
    raise ValueError(f"Missing required vulnerability inputs: {missing}")

df_ml = safe_numeric(df_ml, vuln_inputs)

if df_ml["Elevation_Mean"].isna().sum() > 0:
    missing_elev = df_ml.loc[
        df_ml["Elevation_Mean"].isna(),
        ["Barangay", "barangay_key_strict"]
    ]
    print("Barangays missing Elevation_Mean:")
    display(missing_elev)

    raise ValueError(
        "Elevation_Mean is required for vulnerability computation, "
        "but some barangays have missing elevation."
    )

df_ml["norm_Population_Density_per_Hectare"] = minmax_normalize(
    df_ml["Population_Density_per_Hectare"]
)

df_ml["norm_Population_Change_2020_2024_Pct"] = minmax_normalize(
    df_ml["Population_Change_2020_2024_Pct"]
)

df_ml["norm_Elevation_reverse"] = minmax_reverse(
    df_ml["Elevation_Mean"]
)

vulnerability_components = [
    "norm_Population_Density_per_Hectare",
    "norm_Population_Change_2020_2024_Pct",
    "norm_Elevation_reverse",
]

df_ml["Vulnerability"] = df_ml[vulnerability_components].mean(axis=1)
df_ml["Vulnerability_Elevation_Included"] = True

print("Vulnerability components:", vulnerability_components)
print("Vulnerability includes required reverse-normalized Elevation_Mean.")
print(df_ml["Vulnerability"].describe())

display(
    df_ml[[
        "Barangay",
        "Population_Density_per_Hectare",
        "Population_Change_2020_2024_Pct",
        "Elevation_Mean",
        "norm_Elevation_reverse",
        "Vulnerability",
    ]].head()
)


Vulnerability components: ['norm_Population_Density_per_Hectare', 'norm_Population_Change_2020_2024_Pct', 'norm_Elevation_reverse']
Vulnerability includes required reverse-normalized Elevation_Mean.
count    897.000000
mean       0.341489
std        0.063581
min        0.048382
25%        0.308613
50%        0.336949
75%        0.372616
max        0.630506
Name: Vulnerability, dtype: float64


,Barangay,Population_Density_per_Hectare,Population_Change_2020_2024_Pct,Elevation_Mean,norm_Elevation_reverse,Vulnerability
0,Barangay 1,629.497068,4.140234,2.569270,0.718932,0.327883
1,Barangay 2,439.740638,-11.711712,2.726790,0.701700,0.298873
2,Barangay 3,316.379617,-8.128262,2.581515,0.717593,0.295507
3,Barangay 4,427.181509,1.537552,3.000000,0.671812,0.294023
4,Barangay 5,533.913235,2.562112,3.000000,0.671812,0.303438


## 20. Entropy weighting

**Purpose:** Derive objective weights for the three high-level DPI components (CSI, Exposure, Vulnerability).

**Input:** Component score columns.

**Output:** Entropy values, degree of diversification, and entropy weights (printed).

**Why entropy weighting across component scores:** The thesis weights the three constructs by information content across barangays rather than assigning arbitrary fixed weights. Raw indicators are not entropy-weighted individually in the final method.


In [19]:
component_cols = ["CSI", "Exposure", "Vulnerability"]
entropy_w, entropy_values, diversification_values = entropy_weights(df_ml[component_cols])

weights_table = pd.DataFrame({
    "Component": entropy_w.index,
    "Entropy_Value": entropy_values.values,
    "Diversification_Value": diversification_values.values,
    "Entropy_Weight": entropy_w.values,
})
print("Entropy-derived component weights:")
print(weights_table.to_string(index=False))
print("Weight sum:", float(entropy_w.sum()))
weights_table.to_csv(OUTPUT_DIR / "entropy_component_weights.csv", index=False)


Entropy-derived component weights:
    Component  Entropy_Value  Diversification_Value  Entropy_Weight
          CSI       0.987678               0.012322        0.192859
     Exposure       0.952112               0.047888        0.749503
Vulnerability       0.996317               0.003683        0.057638
Weight sum: 1.0


## 21. DPI computation and scaling

**Purpose:** Combine weighted components into the Disaster Prioritization Index and rescale to 0–100.

`DPI = w_CSI * CSI + w_Exposure * Exposure + w_Vulnerability * Vulnerability`

`DPI_Scaled = minmax(DPI) * 100`


In [20]:
df_ml["DPI"] = (
    entropy_w["CSI"] * df_ml["CSI"]
    + entropy_w["Exposure"] * df_ml["Exposure"]
    + entropy_w["Vulnerability"] * df_ml["Vulnerability"]
)
df_ml["DPI_Scaled"] = minmax_normalize(df_ml["DPI"]) * 100

print("DPI computed and scaled to 0-100.")
print(df_ml[["Barangay", "CSI", "Exposure", "Vulnerability", "DPI", "DPI_Scaled"]].head())
print(df_ml["DPI_Scaled"].describe())


DPI computed and scaled to 0-100.
     Barangay       CSI  Exposure  Vulnerability       DPI  DPI_Scaled
0  Barangay 1  0.250762  0.083278       0.327883  0.129677   15.981407
1  Barangay 2  0.256200  0.046313       0.298873  0.101349   12.312496
2  Barangay 3  0.516438  0.070001       0.295507  0.169098   21.086870
3  Barangay 4  0.388906  0.072191       0.294023  0.146058   18.102935
4  Barangay 5  0.403388  0.059435       0.303438  0.139833   17.296712
count    897.000000
mean      20.710152
std        9.978904
min        0.000000
25%       16.248725
50%       20.122083
75%       24.816529
max      100.000000
Name: DPI_Scaled, dtype: float64


## 22. DPI class generation using tertiles

**Purpose:** Create ordinal priority classes Low / Medium / High from scaled DPI using tertiles (`qcut`).

**Output column:** `DPI_Risk_Class`

**Why DPI-derived classes are ML targets:** Externally observed flood-impact labels are not available in this dataset. Models learn to approximate the deterministic prioritization index classes for methodological comparison — **not** historical flood-event outcomes and **not** real-time flood prediction.

If duplicate bin edges occur, `duplicates='drop'` is used and the approach is reported.


In [21]:
try:
    df_ml["DPI_Risk_Class"] = pd.qcut(
        df_ml["DPI_Scaled"],
        q=3,
        labels=["Low", "Medium", "High"],
        duplicates="drop",
    )
    qcut_note = "Tertile classification via pd.qcut on DPI_Scaled (duplicates='drop')."
except ValueError as exc:
    ranks = df_ml["DPI_Scaled"].rank(method="first")
    df_ml["DPI_Risk_Class"] = pd.qcut(ranks, q=3, labels=["Low", "Medium", "High"])
    qcut_note = (
        f"qcut on raw DPI_Scaled failed ({exc}); used rank-based qcut to force three classes."
    )

print(qcut_note)
print("DPI_Risk_Class distribution:")
print(df_ml["DPI_Risk_Class"].value_counts().to_string())
print("Class proportions (%):")
print((df_ml["DPI_Risk_Class"].value_counts(normalize=True) * 100).round(2).to_string())


Tertile classification via pd.qcut on DPI_Scaled (duplicates='drop').
DPI_Risk_Class distribution:
DPI_Risk_Class
Low       299
Medium    299
High      299
Class proportions (%):
DPI_Risk_Class
Low       33.33
Medium    33.33
High      33.33


## 23. Leakage-free ML feature selection

**Purpose:** Build the predictor matrix while excluding all target- and index-derived fields.

**Excluded (non-exhaustive reminder):** CSI, Exposure, Vulnerability, DPI, DPI_Scaled, DPI_Risk_Class, normalized component intermediates, identifiers, validation flags, geometry, 100-year fields, Relative Population Density, Area_Difference_Pct (QC only).

**Recommended predictors (included when present):**
Flood percentages/escalation/severity, Population_2024, affected population fields, density, population-change %, elevation, and district context flood/density averages plus relative flood risk vs district.

**Why leakage prevention is necessary:** Including DPI components or the class label as predictors would trivially recover the target and invalidate model evaluation as a test of learning from raw/engineered inputs.


In [22]:
target_col = "DPI_Risk_Class"

candidate_features = [
    # Hazard raw/engineered predictors
    "Flood_PCT_5yr",
    "Flood_PCT_25yr",
    "Flood_Escalation_5to25",
    "Flood_Severity_Increase_5to25_Sqm",

    # Exposure raw/engineered predictors
    "Population_2024",
    "Affected_Population_5yr",
    "Affected_Population_25yr",
    "Affected_Population_Increase_5to25",

    # Vulnerability raw predictors
    "Population_Density_per_Hectare",
    "Population_Change_2020_2024_Pct",
    "Elevation_Mean",

    # Context/raw spatial predictors
    "District_Avg_Flood_PCT_5yr",
    "District_Avg_Flood_PCT_25yr",
    "District_Avg_Population_Density_per_Hectare",
    "Relative_Flood_Risk_25yr_vs_District",
]

leakage_or_forbidden = {
    "CSI", "Hazard_Score", "Hazard Score", "CSI Score",
    "Exposure", "Exposure_Score", "Exposure Score",
    "Vulnerability", "Vulnerability_Score", "Vulnerability Score",
    "DPI", "DPI_Scaled", "DPI Score", "DPI Score 0to100",
    "DPI_Risk_Class", "DPI Risk Class",
    "Entropy_Weight", "Entropy_Value", "Diversification_Value",

    # Normalized/index intermediate variables
    "norm_Flood_PCT_5yr",
    "norm_Flood_PCT_25yr",
    "norm_Flood_Escalation_5to25",
    "norm_Flood_Severity_Increase_5to25_Sqm",
    "norm_Affected_Population_5yr",
    "norm_Affected_Population_25yr",
    "norm_Affected_Population_Increase_5to25",
    "norm_Population_Density_per_Hectare",
    "norm_Population_Change_2020_2024_Pct",
    "norm_Elevation_reverse",

    # IDs / labels / geometry / QC-only fields
    "Barangay", "Barangay_No", "psgc_10d", "brgy_code", "city_name",
    "barangay_clean", "barangay_key_strict",
    "Return_Period_Valid_5_25", "flag_25yr_less_than_5yr",
    "geometry",
    "Relative_Population_Density", "Relative Pop Density vs District",
    "Relative Pop Density 2024 vs District",
    "Area_Difference_Pct", "Area Difference Pct",
    "Population_2020",
}

ml_features = [
    c for c in candidate_features
    if c in df_ml.columns and c not in leakage_or_forbidden
]
ml_features = list(dict.fromkeys(ml_features))

if not ml_features:
    raise ValueError("No valid ML features remain after leakage filtering.")

bad = [c for c in ml_features if c in leakage_or_forbidden]
if bad:
    raise ValueError(f"Leakage/forbidden features selected: {bad}")

if "Elevation_Mean" not in ml_features:
    raise ValueError("Elevation_Mean must be included as a raw ML predictor.")

print("Leakage-free ML features:")
for c in ml_features:
    print(" -", c)
print("Feature count:", len(ml_features))

ml_ready_cols = keep_existing(
    ["Barangay", "Barangay_No", "psgc_10d", "city_name"],
    df_ml
) + ml_features + [target_col]

df_ml_ready = df_ml[ml_ready_cols].copy()
df_ml_ready.to_csv(OUT_ML_READY_CSV, index=False)

print(f"Exported ML-ready dataset: {OUT_ML_READY_CSV} shape={df_ml_ready.shape}")

# Leakage control note:
# Raw Elevation_Mean is allowed as a predictor.
# norm_Elevation_reverse and Vulnerability are DPI-derived/intermediate fields and must not be used as predictors.
for leakage_col in [
    "norm_Elevation_reverse",
    "Vulnerability",
    "DPI",
    "DPI_Scaled",
    "CSI",
    "Exposure",
]:
    assert leakage_col not in ml_features, f"Leakage column included in ML features: {leakage_col}"


Leakage-free ML features:
 - Flood_PCT_5yr
 - Flood_PCT_25yr
 - Flood_Escalation_5to25
 - Flood_Severity_Increase_5to25_Sqm
 - Population_2024
 - Affected_Population_5yr
 - Affected_Population_25yr
 - Affected_Population_Increase_5to25
 - Population_Density_per_Hectare
 - Population_Change_2020_2024_Pct
 - Elevation_Mean
Feature count: 11
Exported ML-ready dataset: outputs/manila_ml_ready_dataset.csv shape=(897, 14)


## 24. Train-test split

**Purpose:** Create an 80/20 stratified split by `DPI_Risk_Class` with `random_state=42`.

**Why both split and CV are used:** The held-out test set estimates final generalization once. Cross-validation on the **training set only** supports model selection without touching the test set.


In [23]:
df_model = df_ml[ml_features + [target_col]].dropna(subset=[target_col]).copy()
X = df_model[ml_features]
y = df_model[target_col].astype(str)

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Class encoding:")
for name, code_ in zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)):
    print(f"  {name} -> {code_}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.20, random_state=RANDOM_STATE, stratify=y_encoded
)

print("Train size:", X_train.shape, "Test size:", X_test.shape)
print("Train class distribution:", dict(zip(*np.unique(y_train, return_counts=True))))
print("Test class distribution:", dict(zip(*np.unique(y_test, return_counts=True))))


Class encoding:
  High -> 0
  Low -> 1
  Medium -> 2
Train size: (717, 11) Test size: (180, 11)
Train class distribution: {np.int64(0): np.int64(239), np.int64(1): np.int64(239), np.int64(2): np.int64(239)}
Test class distribution: {np.int64(0): np.int64(60), np.int64(1): np.int64(60), np.int64(2): np.int64(60)}


## 25. Model pipelines

**Purpose:** Define Random Forest, Gradient Boosting, and MLP pipelines with preprocessing inside the pipeline.

- Trees: `SimpleImputer(median) -> Classifier`
- MLP: `SimpleImputer(median) -> StandardScaler -> MLPClassifier`

**Why pipelines:** Imputation/scaling must fit on training folds only. Fitting on the full dataset before splitting would leak test information.


In [24]:
base_models = {
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(random_state=RANDOM_STATE, class_weight="balanced")),
    ]),
    "Gradient Boosting": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", GradientBoostingClassifier(random_state=RANDOM_STATE)),
    ]),
    "MLP": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", MLPClassifier(
            random_state=RANDOM_STATE,
            max_iter=1000,
            early_stopping=True,
        )),
    ]),
}

param_grids = {
    "Random Forest": {
        "model__n_estimators": [200, 300],
        "model__max_depth": [None, 12],
        "model__min_samples_leaf": [1, 2],
    },
    "Gradient Boosting": {
        "model__n_estimators": [100, 200],
        "model__learning_rate": [0.05, 0.1],
        "model__max_depth": [2, 3],
    },
    "MLP": {
        "model__hidden_layer_sizes": [(64, 32), (32,)],
        "model__alpha": [1e-4, 1e-3],
    },
}

print("Model pipelines defined:", list(base_models.keys()))


Model pipelines defined: ['Random Forest', 'Gradient Boosting', 'MLP']


## 26. Cross-validation and hyperparameter tuning

**Purpose:** Stratified 5-fold CV on the **training set** with primary selection metric **F1-macro**, using identical folds across models via a shared `StratifiedKFold`.

**Why F1-macro:** Treats Low/Medium/High classes equally, which is appropriate for priority-class balance assessment.


In [25]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_rows = []
tuned_models = {}
best_params = {}

for name, pipe in base_models.items():
    print(f"\nTuning {name} with GridSearchCV (training set only)...")
    search = GridSearchCV(
        estimator=pipe,
        param_grid=param_grids[name],
        scoring="f1_macro",
        cv=cv,
        n_jobs=-1,
        refit=True,
    )
    search.fit(X_train, y_train)
    tuned_models[name] = search.best_estimator_
    best_params[name] = search.best_params_

    cv_rows.append({
        "Model": name,
        "CV_F1_Macro_Mean": float(search.best_score_),
        "CV_F1_Macro_Std": float(search.cv_results_["std_test_score"][search.best_index_]),
        "Best_Params": str(search.best_params_),
    })
    print(f"Best CV F1-macro: {search.best_score_:.4f}")
    print("Best params:", search.best_params_)

cv_results_df = pd.DataFrame(cv_rows).sort_values("CV_F1_Macro_Mean", ascending=False)
print("\nCross-validation model selection summary:")
print(cv_results_df.to_string(index=False))



Tuning Random Forest with GridSearchCV (training set only)...
Best CV F1-macro: 0.9312
Best params: {'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__n_estimators': 200}

Tuning Gradient Boosting with GridSearchCV (training set only)...
Best CV F1-macro: 0.9343
Best params: {'model__learning_rate': 0.1, 'model__max_depth': 2, 'model__n_estimators': 200}

Tuning MLP with GridSearchCV (training set only)...
Best CV F1-macro: 0.8817
Best params: {'model__alpha': 0.0001, 'model__hidden_layer_sizes': (64, 32)}

Cross-validation model selection summary:
            Model  CV_F1_Macro_Mean  CV_F1_Macro_Std                                                                          Best_Params
Gradient Boosting          0.934322         0.022188     {'model__learning_rate': 0.1, 'model__max_depth': 2, 'model__n_estimators': 200}
    Random Forest          0.931196         0.029691 {'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__n_estimators': 200}
              ML

## 27. Final held-out test evaluation

**Purpose:** Select the best model by training CV F1-macro, ensure it is refit on the full training set (GridSearchCV `refit=True`), then evaluate **once** on the held-out test set.

**Why evaluate the selected model only once on the test set:** Repeated test-set peeking would bias the final generalization estimate. Test metrics are for final reporting, not model selection.

Reported metrics include Accuracy, Balanced Accuracy, per-class Precision/Recall/F1, F1-Macro, F1-Weighted, ROC-AUC OvR Macro (when probabilities exist), Confusion Matrix, and Classification Report.


In [26]:
def evaluate_model(name, model, X_te, y_te, label_encoder):
    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te) if hasattr(model, "predict_proba") else None
    try:
        roc = (
            roc_auc_score(y_te, y_proba, multi_class="ovr", average="macro")
            if y_proba is not None else np.nan
        )
    except Exception:
        roc = np.nan

    precision, recall, f1, support = precision_recall_fscore_support(
        y_te, y_pred, labels=list(range(len(label_encoder.classes_))), zero_division=0
    )
    metrics = {
        "Model": name,
        "Accuracy": accuracy_score(y_te, y_pred),
        "Balanced_Accuracy": balanced_accuracy_score(y_te, y_pred),
        "F1_Macro": f1_score(y_te, y_pred, average="macro"),
        "F1_Weighted": f1_score(y_te, y_pred, average="weighted"),
        "ROC_AUC_OVR_Macro": roc,
    }
    for i, cls in enumerate(label_encoder.classes_):
        metrics[f"Precision_{cls}"] = precision[i]
        metrics[f"Recall_{cls}"] = recall[i]
        metrics[f"F1_{cls}"] = f1[i]
        metrics[f"Support_{cls}"] = support[i]

    cm = confusion_matrix(y_te, y_pred)
    report = classification_report(y_te, y_pred, target_names=label_encoder.classes_)
    return metrics, cm, report, y_pred


best_model_name = cv_results_df.iloc[0]["Model"]
best_model = tuned_models[best_model_name]
print("Best model by training CV F1-macro:", best_model_name)

test_rows = []
confusion_matrices = {}
classification_reports = {}

for name, model in tuned_models.items():
    print(f"\n==== Held-out test evaluation: {name} ====")
    metrics, cm, report, _ = evaluate_model(name, model, X_test, y_test, label_encoder)
    test_rows.append(metrics)
    confusion_matrices[name] = cm
    classification_reports[name] = report
    print(report)
    print("Confusion matrix:\n", cm)

test_results_df = pd.DataFrame(test_rows)
print("\nSelected best model test metrics:")
print(test_results_df.loc[test_results_df["Model"] == best_model_name].T)


Best model by training CV F1-macro: Gradient Boosting

==== Held-out test evaluation: Random Forest ====
              precision    recall  f1-score   support

        High       0.95      0.92      0.93        60
         Low       0.93      0.92      0.92        60
      Medium       0.84      0.88      0.86        60

    accuracy                           0.91       180
   macro avg       0.91      0.91      0.91       180
weighted avg       0.91      0.91      0.91       180

Confusion matrix:
 [[55  0  5]
 [ 0 55  5]
 [ 3  4 53]]

==== Held-out test evaluation: Gradient Boosting ====
              precision    recall  f1-score   support

        High       0.93      0.93      0.93        60
         Low       0.95      0.93      0.94        60
      Medium       0.87      0.88      0.88        60

    accuracy                           0.92       180
   macro avg       0.92      0.92      0.92       180
weighted avg       0.92      0.92      0.92       180

Confusion matrix:
 [[5

## 28. Model comparison table

**Purpose:** Combine training CV selection metrics and held-out test metrics for all three models into one comparison table.

**Output:** `model_comparison_results.csv`


In [27]:
comparison_df = cv_results_df.merge(test_results_df, on="Model", how="left")
comparison_df = comparison_df.sort_values(["CV_F1_Macro_Mean", "F1_Macro"], ascending=False)
comparison_df.to_csv(OUT_MODEL_COMPARISON_CSV, index=False)
print("Model comparison table:")
print(comparison_df.to_string(index=False))
print(f"Saved: {OUT_MODEL_COMPARISON_CSV}")


Model comparison table:
            Model  CV_F1_Macro_Mean  CV_F1_Macro_Std                                                                          Best_Params  Accuracy  Balanced_Accuracy  F1_Macro  F1_Weighted  ROC_AUC_OVR_Macro  Precision_High  Recall_High  F1_High  Support_High  Precision_Low  Recall_Low   F1_Low  Support_Low  Precision_Medium  Recall_Medium  F1_Medium  Support_Medium
Gradient Boosting          0.934322         0.022188     {'model__learning_rate': 0.1, 'model__max_depth': 2, 'model__n_estimators': 200}  0.916667           0.916667  0.916848     0.916848           0.986343        0.933333     0.933333 0.933333            60       0.949153    0.933333 0.941176           60          0.868852       0.883333   0.876033              60
    Random Forest          0.931196         0.029691 {'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__n_estimators': 200}  0.905556           0.905556  0.906121     0.906121           0.989769        0.948276     0.91666

## 29. Feature importance extraction

**Purpose:** Extract impurity-based feature importance for Random Forest and Gradient Boosting.

**Interpretation:** Values are **model-based importance** for approximating DPI-derived classes. They do **not** imply causal drivers of real-world flooding.

If the selected best model is MLP, tree-model importances are still reported; standard MLP has no direct impurity-based importance.


In [28]:
importance_frames = {}
for name in ["Random Forest", "Gradient Boosting"]:
    model = tuned_models[name]
    clf = model.named_steps["model"]
    if hasattr(clf, "feature_importances_"):
        imp_df = pd.DataFrame({
            "Feature": ml_features,
            "Importance": clf.feature_importances_,
            "Model": name,
        }).sort_values("Importance", ascending=False)
        importance_frames[name] = imp_df
        out_path = OUTPUT_DIR / f"feature_importance_{name.lower().replace(' ', '_')}.csv"
        imp_df.to_csv(out_path, index=False)
        print(f"\nFeature importance ({name}) — model-based only:")
        print(imp_df.to_string(index=False))

if best_model_name == "MLP":
    print(
        "\nNote: Best model is MLP. Standard MLPClassifier does not provide "
        "impurity-based feature importance; tree-model importances above are reported for interpretation support only."
    )



Feature importance (Random Forest) — model-based only:
                           Feature  Importance         Model
          Affected_Population_25yr    0.309513 Random Forest
           Affected_Population_5yr    0.160888 Random Forest
                   Population_2024    0.145088 Random Forest
                    Flood_PCT_25yr    0.138182 Random Forest
                     Flood_PCT_5yr    0.066725 Random Forest
Affected_Population_Increase_5to25    0.060536 Random Forest
 Flood_Severity_Increase_5to25_Sqm    0.030526 Random Forest
            Flood_Escalation_5to25    0.027791 Random Forest
    Population_Density_per_Hectare    0.025165 Random Forest
                    Elevation_Mean    0.023439 Random Forest
   Population_Change_2020_2024_Pct    0.012147 Random Forest

Feature importance (Gradient Boosting) — model-based only:
                           Feature  Importance             Model
          Affected_Population_25yr    0.601811 Gradient Boosting
                    Fl

## 30. Barangay-level prediction and ranking output

**Purpose:** Apply the selected best model to all barangays; attach predicted class, confidence, DPI scores, and a DPI-based rank.

**Output:** `barangay_flood_risk_predictions.csv`

**Interpretation reminder:** These are planning-support priority outputs derived from an index-based prototype, not official flood warnings.


In [29]:
X_all = df_ml[ml_features].copy()
df_ml["ML_Predicted_Class_Encoded"] = best_model.predict(X_all)
df_ml["ML_Predicted_Risk_Class"] = label_encoder.inverse_transform(df_ml["ML_Predicted_Class_Encoded"])
if hasattr(best_model, "predict_proba"):
    df_ml["ML_Prediction_Confidence"] = best_model.predict_proba(X_all).max(axis=1)
else:
    df_ml["ML_Prediction_Confidence"] = np.nan
df_ml["Best_Model"] = best_model_name
df_ml["DPI_Rank"] = df_ml["DPI_Scaled"].rank(ascending=False, method="min").astype(int)

pred_cols = keep_existing(
    [
        "Barangay", "Barangay_No", "psgc_10d", "city_name",
        "Population_2020", "Population_2024",
        "Population_Density_per_Hectare",
        "Population_Change_2020_2024_Pct",
        "Elevation_Mean", "Elevation_Median", "Elevation_Min", "Elevation_Max",
        "Flood_PCT_5yr", "Flood_PCT_25yr",
        "CSI", "Exposure", "Vulnerability",
        "DPI", "DPI_Scaled", "DPI_Risk_Class", "DPI_Rank",
        "ML_Predicted_Risk_Class", "ML_Prediction_Confidence", "Best_Model",
        "Vulnerability_Elevation_Included",
    ],
    df_ml,
)
predictions_out = df_ml[pred_cols].sort_values("DPI_Rank").reset_index(drop=True)
predictions_out.to_csv(OUT_PREDICTIONS_CSV, index=False)
print(f"Saved barangay predictions: {OUT_PREDICTIONS_CSV}")
print(predictions_out.head(10))


Saved barangay predictions: outputs/barangay_flood_risk_predictions.csv
       Barangay  Barangay_No  Population_2020  Population_2024  Population_Density_per_Hectare  Population_Change_2020_2024_Pct  Elevation_Mean  \
0  Barangay 310          310            13602          13761.0                     1590.486639                         1.168946        2.000000   
1  Barangay 628          628            15202          16279.0                      884.899544                         7.084594        1.408893   
2  Barangay 649          649            64750          69769.0                      632.795155                         7.751351        0.625217   
3  Barangay 598          598            12027          10510.0                     1368.152765                       -12.613287        2.567198   
4  Barangay 648          648             5634           9086.0                     2149.392097                        61.270856        1.249852   
5   Barangay 20           20            45772 

## 31. Export of final model and outputs

**Purpose:** Serialize the best pipeline and confirm all required thesis outputs exist.

**Outputs:**
1. `MetroManila_Barangay_Flood_Area_Validated.csv`
2. `manila-city-flood-population_enriched_NEW.csv`
3. `manila-city-flood-population_enriched_NEW.xlsx`
4. `manila_ml_ready_dataset.csv`
5. `model_comparison_results.csv`
6. `barangay_flood_risk_predictions.csv`
7. `best_flood_risk_model_pipeline.joblib`


In [30]:
joblib.dump(
    {
        "pipeline": best_model,
        "label_encoder": label_encoder,
        "features": ml_features,
        "best_model_name": best_model_name,
        "best_params": best_params.get(best_model_name),
        "random_state": RANDOM_STATE,
    },
    OUT_BEST_MODEL_JOBLIB,
)

required_outputs = [
    OUT_FLOOD_CSV,
    OUT_ENRICHED_CSV,
    OUT_ENRICHED_XLSX,
    OUT_ML_READY_CSV,
    OUT_MODEL_COMPARISON_CSV,
    OUT_PREDICTIONS_CSV,
    OUT_BEST_MODEL_JOBLIB,
]

print("\nOutput file checklist:")
for path in required_outputs:
    status = "OK" if path.exists() else "MISSING"
    print(f"  [{status}] {path}")

missing_outputs = [p for p in required_outputs if not p.exists()]
if missing_outputs:
    critical = [p for p in missing_outputs if p.suffix.lower() != ".xlsx"]
    if critical:
        raise FileNotFoundError(f"Missing required outputs: {critical}")
    print("WARNING: Excel output missing (non-critical if openpyxl unavailable).")

print("\nBest model:", best_model_name)
print("Feature count:", len(ml_features))
print("Barangay count:", len(df_ml))

print("Elevation CSV integrated:", "Elevation_Mean" in df_ml.columns and df_ml["Elevation_Mean"].notna().any())
if "Elevation_Mean" in df_ml.columns:
    print("Missing elevation values:", int(df_ml["Elevation_Mean"].isna().sum()))
    if REQUIRE_ELEVATION:
        assert df_ml["Elevation_Mean"].isna().sum() == 0, "Missing required elevation values."
assert df_ml["DPI_Risk_Class"].nunique() == 3, "DPI risk classification did not produce three classes."



Output file checklist:
  [OK] outputs/MetroManila_Barangay_Flood_Area_Validated.csv
  [OK] outputs/manila-city-flood-population_enriched_NEW.csv
  [OK] outputs/manila-city-flood-population_enriched_NEW.xlsx
  [OK] outputs/manila_ml_ready_dataset.csv
  [OK] outputs/model_comparison_results.csv
  [OK] outputs/barangay_flood_risk_predictions.csv
  [OK] outputs/best_flood_risk_model_pipeline.joblib

Best model: Gradient Boosting
Feature count: 11
Barangay count: 897
Elevation CSV integrated: True
Missing elevation values: 0


## 32. Final reproducibility and limitation notes

**Reproducibility**
- Fixed `random_state=42` for split, CV folds, and model estimators.
- Preprocessing is confined to sklearn pipelines / GridSearchCV folds.
- Model selection uses training CV F1-macro only; held-out test is evaluated after selection.
- Intermediate and final outputs are written to the configured `OUTPUT_DIR`.
- Elevation is supplied through the required QGIS Zonal Statistics output file.

**Methodological limitations**
- LiPAD flood layers are modeled hazard surfaces, not observed event inundation maps.
- The 100-year layer is excluded from DPI/ML due to return-period consistency concerns.
- DPI risk classes are index-derived proxy labels, not ground-truth flood impacts.
- Supervised metrics measure agreement with the deterministic DPI classification, not real-world forecast skill.
- Elevation is represented as barangay-level summary elevation and does not capture micro-scale drainage, road depressions, drainage capacity, land subsidence, or building-level exposure.
- Population Change 2020–2024 (%) is a four-year observed change, not an annualized growth rate.
- District features depend on the administrative `city_name` district partition present in the base dataset.

**Use statement**
Prototype outputs support barangay-level planning prioritization research for the City of Manila. They must not be interpreted as official flood warnings, evacuation orders, or real-time flood predictions.

In [31]:
print("=" * 72)
print("Manila flood risk complete pipeline finished.")
print(f"Barangays: {len(df_ml)} (expected {EXPECTED_BARANGAY_COUNT})")
print(f"Best model: {best_model_name}")
print(f"Outputs directory: {OUTPUT_DIR}")
print("=" * 72)


Manila flood risk complete pipeline finished.
Barangays: 897 (expected 897)
Best model: Gradient Boosting
Outputs directory: outputs


### QGIS Elevation Data Update

The vulnerability component uses barangay-level mean elevation extracted from a DEM/DTM raster using QGIS Zonal Statistics. The Manila barangay polygon layer served as the zone layer, while the DEM/DTM raster served as the elevation raster.

The QGIS output fields are standardized as `Elevation_Count`, `Elevation_Mean`, `Elevation_Median`, `Elevation_Min`, and `Elevation_Max`.

For the DPI, `Elevation_Mean` is used as the physical susceptibility indicator. It is reverse-normalized so that lower terrain elevation contributes to higher vulnerability. Raw `Elevation_Mean` is permitted as an ML predictor, while `norm_Elevation_reverse` and `Vulnerability` are excluded from the ML feature set to prevent data leakage.
